# Stage 3 v2 — Hard Fine-tuning với 5 cải tiến

**5 cải tiến tăng khả thi từ ~50% → ~80%:**
1. Mix source: 50% UTKFace Asian + 50% FFHQ subset (anti-forgetting)
2. Mix degradation: 70% hard + 30% medium (anti-shock)
3. EMA weights (decay 0.999) cho Generator
4. R1 gradient penalty (lazy, every 16 step)
5. Save every 500 iter (10 candidates)

**Datasets cần attach:**
1. Code project (chứa `trainer_stage3.py` v2 + `stage3_hard.yaml` + `mixed_dataset_utils.py`)
2. `vn-pretrained-weights-v1` — Real-ESRGAN + GFPGAN
3. `datasets12345` — Stage 2 best.pth
4. `jangedoo/utkface-new` — UTKFace ~24k (sẽ filter race=2)
5. `xhlulu/flickrfaceshq-dataset-nvidia-resized-256px` — FFHQ 70k (sẽ subsample 10k)
6. `badasstechie/celebahq-resized-256x256` — CelebA-HQ (val)


## 1. Verify datasets


In [ ]:
from pathlib import Path
print("=== Kaggle Input Datasets ===")
for p in sorted(Path('/kaggle/input').iterdir()):
    n = sum(1 for _ in p.rglob('*') if _.is_file())
    print(f"  {p.name:<45} ({n} files)")

## 2. Setup code


In [ ]:
import os, sys, shutil
from pathlib import Path

DST = '/kaggle/working/vn-old-portrait-restoration'
SRC = None
for p in Path('/kaggle/input').iterdir():
    hits = list(p.rglob('src/training/trainer_stage3.py'))
    if hits:
        SRC = str(hits[0].parent.parent.parent)
        break
assert SRC, "Code project with trainer_stage3.py not found"

if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
os.chdir(DST)
sys.path.insert(0, DST)
print(f"✓ Code: {SRC} → {DST}")

# Verify v2 files
required = [
    'src/training/trainer_stage3.py',
    'src/utils/mixed_dataset_utils.py',
    'configs/stage3_hard.yaml',
]
for f in required:
    assert Path(f).exists(), f"MISSING: {f}"
    print(f"  ✓ {f}")
print("\n✓ All v2 source files present")

## 3. Dependencies + GPU


In [ ]:
!pip install -q einops basicsr facexlib gfpgan
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name}, VRAM: {g.total_memory/1e9:.1f} GB")

## 4. Find paths


In [ ]:
from pathlib import Path
import re

# Stage 2 ckpt
stage2_list = [p for p in Path('/kaggle/input').glob('**/best.pth')
               if p.stat().st_size >= 100_000_000]
STAGE2_CKPT = str(stage2_list[0])

# Pretrained
re_list = list(Path('/kaggle/input').rglob('RealESRGAN_x4plus.pth'))
gf_list = list(Path('/kaggle/input').rglob('GFPGANv1.4.pth'))
REAL_ESRGAN = str(re_list[0]) if re_list else ""
GFPGAN_PRE = str(gf_list[0]) if gf_list else ""

# UTKFace root (filename pattern age_gender_race_time.jpg)
UTKFACE_ROOT = ""
utk_pattern = re.compile(r'^\d+_\d+_\d+_\d+')
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir():
        sample = list(p.glob('*.jpg'))[:5]
        if len(sample) >= 3 and all(utk_pattern.match(f.name) for f in sample):
            n_match = sum(1 for f in p.glob('*.jpg') if utk_pattern.match(f.name))
            if n_match > 5000:
                UTKFACE_ROOT = str(p)
                break

# FFHQ (~70k png)
FFHQ_DIR = ""
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir():
        n_png = sum(1 for _ in p.glob('*.png'))
        if n_png > 50000:
            FFHQ_DIR = str(p)
            break

# CelebA-HQ
CELEBA = ""
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir() and 'celeb' in p.name.lower():
        n_img = sum(1 for _ in p.rglob('*.jpg')) + sum(1 for _ in p.rglob('*.png'))
        if n_img > 1000:
            CELEBA = str(p)
            break

print(f"Stage 2:       {STAGE2_CKPT} ({Path(STAGE2_CKPT).stat().st_size/1e6:.0f} MB)")
print(f"Real-ESRGAN:   {REAL_ESRGAN}")
print(f"GFPGAN:        {GFPGAN_PRE}")
print(f"UTKFace root:  {UTKFACE_ROOT}")
print(f"FFHQ:          {FFHQ_DIR}")
print(f"CelebA val:    {CELEBA}")

for name, path in [('UTKFace', UTKFACE_ROOT), ('FFHQ', FFHQ_DIR), ('CelebA', CELEBA)]:
    assert path and Path(path).exists(), f"{name} missing"
print("\n✓ All paths OK")

## 5. FILTER UTKFace → chỉ Đông Á (race=2)

UTKFace filename: `[age]_[gender]_[race]_[time].jpg`
- race: 0=White, 1=Black, **2=Asian (Đông Á)**, 3=Indian, 4=Others

Filter race=2 → ~3,400 ảnh Đông Á thuần.


In [ ]:
import os, re
from pathlib import Path

UTKFACE_ASIAN = '/kaggle/working/utkface_asian'
os.makedirs(UTKFACE_ASIAN, exist_ok=True)
for f in Path(UTKFACE_ASIAN).iterdir():
    if f.is_symlink() or f.is_file():
        f.unlink()

pattern = re.compile(r'^(\d+)_(\d+)_(\d+)_')
count_by_race = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
asian_count = 0

for img_path in Path(UTKFACE_ROOT).rglob('*.jpg'):
    m = pattern.match(img_path.name)
    if m:
        try:
            race = int(m.group(3))
        except ValueError:
            continue
        if 0 <= race <= 4:
            count_by_race[race] += 1
            if race == 2:
                target = Path(UTKFACE_ASIAN) / img_path.name
                if not target.exists():
                    try:
                        target.symlink_to(img_path.resolve())
                        asian_count += 1
                    except OSError:
                        import shutil
                        shutil.copy2(img_path, target)
                        asian_count += 1

print("=== UTKFace Distribution ===")
labels = {0: 'White', 1: 'Black', 2: 'Asian (Đông Á)', 3: 'Indian', 4: 'Others'}
total = sum(count_by_race.values())
for r, n in count_by_race.items():
    pct = n / total * 100 if total else 0
    mark = "  ← USED" if r == 2 else ""
    print(f"  {labels[r]:<20} {n:>5}  ({pct:5.1f}%){mark}")
print(f"\nĐông Á filtered: {asian_count}")
print(f"Output:          {UTKFACE_ASIAN}")
assert asian_count > 1000, f"Asian too few: {asian_count}"
print(f"\n✓ {asian_count} Đông Á images ready")

## 6. Build config


In [ ]:
import yaml

with open('configs/stage3_hard.yaml') as f:
    config = yaml.safe_load(f)

config['model']['real_esrgan']['pretrained'] = REAL_ESRGAN
config['model']['gfpgan']['pretrained'] = GFPGAN_PRE
config['model']['stage2_checkpoint'] = STAGE2_CKPT
config['data']['asian_dir'] = UTKFACE_ASIAN
config['data']['ffhq_dir'] = FFHQ_DIR
config['data']['val_dir'] = CELEBA

ffhq_max = config['data']['ffhq_max_samples']
total_train = asian_count + min(ffhq_max, sum(1 for _ in Path(FFHQ_DIR).glob('*.png')))

print("=" * 60)
print("STAGE 3 v2 CONFIG — FINAL")
print("=" * 60)
print(f"\n[Data] — MIX 2 sources × 2 deg levels")
print(f"  Asian (UTKFace race=2):  {asian_count:>5}")
print(f"  FFHQ subset:             {ffhq_max:>5}")
print(f"  Mix ratio (Asian/FFHQ):  {config['data']['ratio_asian']*100:.0f}% / {(1-config['data']['ratio_asian'])*100:.0f}%")
print(f"  Mix ratio (hard/medium): {config['data']['ratio_hard']*100:.0f}% / {(1-config['data']['ratio_hard'])*100:.0f}%")
print(f"  Val:                     CelebA-HQ × {config['data']['val_degradation_levels']}")
print(f"\n[Model]")
print(f"  ESRGAN unfreeze:  {config['model']['real_esrgan']['unfreeze_n']}/23 RRDB (full)")
print(f"  GFPGAN:           {config['model']['gfpgan']['unfreeze_strategy']}")
print(f"  Fusion blocks:    {config['model']['transformer_fusion']['num_blocks']} (Stage 2: 2)")
print(f"\n[Train v2 features]")
print(f"  EMA:              {'ON (decay=' + str(config['train']['ema']['decay']) + ')' if config['train']['ema']['enable'] else 'OFF'}")
print(f"  R1 penalty:       {'ON (every ' + str(config['loss']['r1_penalty']['every_n_steps']) + ', w=' + str(config['loss']['r1_penalty']['weight']) + ')' if config['loss']['r1_penalty']['enable'] else 'OFF'}")
print(f"  Save every:       {config['train']['save_every']} iter (10 candidates)")
print(f"  Val every:        {config['train']['val_every']} iter")
print(f"  Freeze D:         {config['train']['freeze_d_first_iters']} iter")
print(f"\n[Training]")
print(f"  Iters:            {config['train']['total_iters']}")
print(f"  Warmup:           {config['optim']['scheduler']['warmup_iters']}")
print(f"  LR backbone:      {config['optim']['lr']}")
print(f"  LR deg_encoder:   {config['optim']['lr_deg_encoder']}")
print(f"  LR fusion:        {config['optim']['lr_new']}")
print(f"  LR discriminator: {config['optim']['lr_d']}")
print(f"\n[Loss weights]")
for k, v in config['loss'].items():
    if isinstance(v, dict) and v.get('enable'):
        print(f"  {k:<25} {v.get('weight')}")

## 7. Patch forward_generator (skip x4 upsample)


In [ ]:
import torch
import torch.nn.functional as F
from src.training.trainer_stage3 import Stage3Trainer

def forward_generator_lowvram(self, lq):
    deg_pred = self.deg_encoder(lq)
    feat = self.bg_net.conv_first(lq)
    feat_skip = feat
    for block in self.bg_net.body:
        if hasattr(block, 'film'):
            feat = block(feat, deg_pred)
        else:
            feat = block(feat)
    feat = self.bg_net.conv_body(feat)
    feat = feat + feat_skip
    bg_out = self.bg_net.conv_last(self.bg_net.lrelu(self.bg_net.conv_hr(feat)))
    bg_out = bg_out.clamp(0, 1)
    face_input = F.interpolate(lq, size=(512, 512), mode='bilinear', align_corners=False)
    face_512 = self.face_net(face_input, deg_pred)
    face_out = F.interpolate(face_512, size=lq.shape[-2:], mode='bilinear', align_corners=False)
    final = self.fusion(face_out, bg_out).clamp(0, 1)
    return {'deg_pred': deg_pred, 'bg_out': bg_out, 'face_out': face_out, 'final': final}

Stage3Trainer.forward_generator = forward_generator_lowvram
print("✓ Patched forward_generator")

## 8. Train Stage 3 v2

**Monitoring:**
- OOM iter 0 → Stop, sửa cell 6: `config['model']['gfpgan']['unfreeze_strategy'] = 'last_n'`, re-run từ cell 6
- EMA val_loss thường thấp hơn raw 0.05-0.1
- R1 only mỗi 16 step, hiển thị trong D log
- Save Version Kaggle mỗi 1-2h


In [ ]:
trainer = Stage3Trainer(config)
# Resume: trainer = Stage3Trainer(config, resume='/kaggle/input/.../iter_XXXX.pth')
trainer.train()

## 9. Verify


In [ ]:
from pathlib import Path
for d in [Path('/kaggle/working/checkpoints/stage3'),
          Path('/kaggle/working/ckpt_backup_stage3')]:
    print(f"\n{d}")
    if d.exists():
        for f in sorted(d.iterdir()):
            print(f"  {f.name:<25} {f.stat().st_size/1e6:>7.1f} MB")

In [ ]:
import torch
best = Path('/kaggle/working/ckpt_backup_stage3/best.pth')
if best.exists():
    ckpt = torch.load(best, map_location='cpu', weights_only=False)
    s2 = ckpt.get('stage2_baseline_val_loss', 0)
    s3 = ckpt.get('best_val_loss', 0)
    print(f"Best iter:       {ckpt.get('iter')}")
    print(f"Stage 2 base:    {s2:.4f}")
    print(f"Stage 3 best:    {s3:.4f}")
    print(f"Improvement:     {s2 - s3:+.4f} ({'✓ BETTER' if s2 > s3 else '✗ WORSE'})")
    print(f"Has EMA:         {'ema' in ckpt}")
    print(f"Fusion keys:     {len(ckpt['fusion'])} (expect ~76 for 4 blocks)")